# DataFrame ビルダ: クエリを Python のオブジェクトとして扱う

`db.table(...)` は、メソッド呼び出しで組み立てる**遅延**クエリの入口です。SQL の文字列は
書きません。`.collect()` のような終端の呼び出しまで、何も走りません。

正体はコンパイラで、第2のエンジンではありません。どの動詞も `db.sql()` を通る SQL に落ちる
ので、組み立てたクエリが見るセッションもテーブル関数もバージョンのピン留めも、手で書いた
文字列の場合とまったく同じです。`.sql()` を呼べば、生成されたものがそのまま見えます。

リサーチのデスクにとっての見返りは、クエリを生成できることです。ウィンドウや列をループで
掃くファクターライブラリは、いまなら f-string で SQL を作っているでしょう。クオートの
バグと `'` のインジェクションが住んでいるのはそこです。ビルダなら識別子のクオートは1か所
に集まり、組み立て途中のパイプラインは、持ち回して伸ばして再利用できる普通の Python の値
になります。

In [1]:
import h5i_db
from h5i_db import col, count_star, lit, sql_expr, time_bucket, vwap, when

import cookbook_utils as cu

db = h5i_db.Database(cu.fresh_db("00_dataframe_builder"), create=True)

## データ

テーブルは2つです。ビルダの動詞も同じ線で分かれるからです。`cu.make_trades` のティック
レベルの `trades` は、バケット化と集約を試す側です。

| 列 | 型 | 意味 |
| --- | --- | --- |
| `ts` | `timestamp[us, tz=UTC]` | 約定時刻、昇順 |
| `symbol` | `string` | 銘柄コード |
| `price` | `float64` | 約定価格 |
| `size` | `int64` | 約定株数 |
| `exchange` | `string` | 報告した取引所 |
| `side` | `string` | `B` は買い主導、`S` は売り主導 |

In [2]:
trades = cu.make_trades(symbols=["AAPL", "MSFT", "NVDA"], days=3, trades_per_day=20_000)
print(f"trades: {trades.num_rows:,} rows x {trades.num_columns} columns")
trades.to_pandas().head()

trades: 195,277 rows x 6 columns


,ts,symbol,price,size,exchange,side
0,2026-06-01 13:30:00.111237+00:00,NVDA,319.22,1,NASDAQ,S
1,2026-06-01 13:30:00.168881+00:00,NVDA,319.27,1,ARCA,B
2,2026-06-01 13:30:00.204185+00:00,NVDA,319.28,1,IEX,B
3,2026-06-01 13:30:00.329485+00:00,MSFT,363.06,1,NASDAQ,B
4,2026-06-01 13:30:00.381450+00:00,NVDA,319.22,300,ARCA,S


`cu.make_daily_prices` の日足 OHLCV パネル、50銘柄×500セッションのほうは、ウィンドウと
クロスセクションの動詞を試す側です。列は `ts`、`symbol`、`open`、`high`、`low`、`close`、
`volume` です。

In [3]:
prices = cu.make_daily_prices(days=500)  # 50 names x 500 sessions
print(f"prices: {prices.num_rows:,} rows x {prices.num_columns} columns")
prices.to_pandas().head()

prices: 25,000 rows x 7 columns


,ts,symbol,open,high,low,close,volume
0,2023-01-02 20:00:00+00:00,STK000,25.80,25.88,25.73,25.86,451179
1,2023-01-02 20:00:00+00:00,STK001,240.29,241.67,240.06,240.88,208895
2,2023-01-02 20:00:00+00:00,STK002,243.93,244.20,242.34,243.66,388381
3,2023-01-02 20:00:00+00:00,STK003,197.96,198.65,196.13,198.47,776701
4,2023-01-02 20:00:00+00:00,STK004,77.51,78.01,77.39,77.51,348154


In [4]:
db.create_table("trades", trades.schema, time_column="ts", sort_key=["ts", "symbol"])
db.append("trades", trades)

db.create_table("prices", prices.schema, time_column="ts", sort_key=["ts", "symbol"])
db.append("prices", prices)

db.tables()

['prices', 'trades']

## 1. フレームとは、まだ走っていないクエリのこと

`db.table("trades")` はテーブル全体を出発点にします。動詞は**新しい**フレームを返すので、
何も書き換わりませんし、途中まで組んだパイプラインはそのまま再利用できます。`.sql()` が
それを描き出し、`.collect()` が走らせます。

In [5]:
liquid = db.table("trades").filter(col("symbol").is_in(["AAPL", "NVDA"]))
print(liquid.sql())

SELECT *
FROM "trades"
WHERE "symbol" IN ('AAPL', 'NVDA')


定番の OHLCV 集計を組んでみます。`group_by(...).agg(...)` は集約と並べてキーも射影します。
`.first("ts")` と `.last("ts")` は `first_value(x ORDER BY ts)` の定型で、自己結合なしに
バーの始値と終値を出します。

In [6]:
bars = (
    db.table("trades")
    .group_by(time_bucket("1m", col("ts")).alias("bar"), "symbol")
    .agg(
        col("price").first("ts").alias("open"),
        col("price").max().alias("high"),
        col("price").min().alias("low"),
        col("price").last("ts").alias("close"),
        col("size").sum().alias("volume"),
        vwap(col("price"), col("size")).alias("vwap"),
    )
    .sort(["bar", "symbol"])
)
print(bars.sql())

SELECT time_bucket('1m', "ts") AS "bar", "symbol", first_value("price" ORDER BY "ts") AS "open", max("price") AS "high", min("price") AS "low", last_value("price" ORDER BY "ts") AS "close", sum("size") AS "volume", vwap("price", "size") AS "vwap"
FROM "trades"
GROUP BY "bar", "symbol"
ORDER BY "bar", "symbol"


In [7]:
bars.to_pandas().head(6)

,bar,symbol,open,high,low,close,volume,vwap
0,2026-06-01 13:30:00+00:00,AAPL,265.05,265.23,265.01,265.17,10308,265.137426
1,2026-06-01 13:30:00+00:00,MSFT,363.06,363.17,362.76,363.12,9544,362.944815
2,2026-06-01 13:30:00+00:00,NVDA,319.22,319.43,318.97,319.29,12402,319.220094
3,2026-06-01 13:31:00+00:00,AAPL,265.16,265.28,265.13,265.22,11005,265.199838
4,2026-06-01 13:31:00+00:00,MSFT,363.11,363.11,362.73,362.75,10846,362.922534
5,2026-06-01 13:31:00+00:00,NVDA,319.38,319.38,318.78,319.31,10606,319.094032


## 2. 式

`col(name)` が列、`lit(value)` が定数で、あとは算術と比較で積み上げます。早めに出会って
おきたい罠が2つあります。

- Python では `and`、`or`、`not` を多重定義できないので、真偽の論理は `&`、`|`、`~` を
  使います。これらは比較より*強く*結合するので、比較のたびに括弧が要ります。
- 式は Python ではなく **SQL** の意味論を保ちます。整数列どうしの `/` は整数除算です。
  真の除算が欲しければキャストしてください。

In [8]:
signed = (
    db.table("trades")
    .filter((col("price") > 0) & (col("size") >= 100))
    .select(
        "ts",
        "symbol",
        "price",
        notional=col("price") * col("size"),
        lots=col("size").cast("DOUBLE") / 100,
        direction=when(col("side") == "B").then(lit(1)).otherwise(lit(-1)),
    )
)
print(signed.sql())

SELECT "ts", "symbol", "price", "price" * "size" AS "notional", CAST("size" AS DOUBLE) / 100 AS "lots", CASE WHEN "side" = 'B' THEN 1 ELSE -1 END AS "direction"
FROM "trades"
WHERE "price" > 0 AND "size" >= 100


In [9]:
signed.to_pandas().head(4)

,ts,symbol,price,notional,lots,direction
0,2026-06-02 13:53:43.548327+00:00,NVDA,317.02,95106.0,3.0,-1
1,2026-06-02 13:53:43.792285+00:00,AAPL,267.76,26776.0,1.0,-1
2,2026-06-02 13:53:43.894350+00:00,AAPL,267.75,187425.0,7.0,-1
3,2026-06-02 13:53:44.009965+00:00,NVDA,317.17,31717.0,1.0,1


識別子は常にクオートされるので、大文字小文字も残ります。`col("Symbol")` は `Symbol` という
名前のフィールドを見つけます。素の SQL なら小文字に畳まれるところです。そして文字列
リテラルは常に文字列で、構文になることはありません。

In [10]:
print(db.table("trades").filter(col("symbol") == "'; DROP TABLE trades; --").sql())

SELECT *
FROM "trades"
WHERE "symbol" = '''; DROP TABLE trades; --'


## 3. パイプラインが SQL になるまで

たいていのパイプラインは1つの平らな `SELECT` にコンパイルされます。独立した `with_columns`
は1つにまとまりますし、*ベース*の列に対する絞り込みは同じ `WHERE` に残ります。

前の段が**計算した**列を読む段は、自分の階層を持ちます。SQL は `WHERE` を、選択リストの
隣の項目ではなく `FROM` に対して解決するからです。集約と `LIMIT` と `DISTINCT` も階層を
閉じます。後に続くものは、その出力に対して働くからです。

In [11]:
movers = (
    db.table("prices")
    .with_columns(ret=col("close") / col("open") - 1)
    .filter(col("ret") > 0.01)  # reads a computed column -> subquery
    .sort("ret", descending=True)
    .limit(5)
)
print(movers.sql())

SELECT *
FROM (
  SELECT *, "close" / "open" - 1 AS "ret"
  FROM "prices"
) AS "_s1"
WHERE "ret" > 0.01
ORDER BY "ret" DESC
LIMIT 5


In [12]:
movers.to_pandas()

,ts,symbol,open,high,low,close,volume,ret
0,2023-07-06 20:00:00+00:00,STK003,192.62,195.55,192.03,195.05,514942,0.012616
1,2024-08-14 20:00:00+00:00,STK020,131.80,133.81,131.03,133.38,270545,0.011988
2,2023-01-09 20:00:00+00:00,STK016,30.51,30.94,30.42,30.85,480125,0.011144
3,2023-01-06 20:00:00+00:00,STK008,96.41,97.76,96.36,97.47,339522,0.010995
4,2023-07-04 20:00:00+00:00,STK043,342.13,347.28,341.83,345.81,292791,0.010756


階層が*どこで*閉じるかを体で覚えることが、いちばん効きます。次の段が何を見られるかを決める
のがそれだからです。

パイプラインが平らなあいだは、動詞はまだベーステーブルに届きます。
`select("ts", "symbol").sort("close")` はちゃんと解決します。SQL の `ORDER BY` が読むのは
`FROM` だからです。集約が階層を閉じてしまうと、その列は本当に消えます。エンジンもそう
言います。

In [13]:
try:
    db.table("prices").group_by("symbol").agg(count_star().alias("n")).sort("close").collect()
except h5i_db.H5iError as e:
    print(f"{type(e).__name__}: {str(e)[:180]}")

H5iError: [query] Error during planning: Column in ORDER BY must be in GROUP BY or an aggregate function: While expanding wildcard, column "prices.close" must appear in the GROUP BY clause o


## 4. 見返り: 生成するクエリ

ビルダが場所代を稼ぐのはここです。複数のルックバックを掃く処理が、文字列の切り貼りでは
なくフレームに対する Python のループになり、しかも各フレームは持てて名前を付けられて
再利用できる値です。下では、価格とそれ自身の移動平均との乖離を3つのウィンドウで出します。

ローリングのメソッドは `window` と `order_by`、任意で `partition_by` を取ります。SQL の
`rolling_avg` 糖衣と違って本物の `PARTITION BY` を持つので、複数銘柄のテーブルでも銘柄を
混ぜることは**ありません**。

In [14]:
WINDOWS = (5, 20, 60)

base = db.table("prices").filter(col("symbol").is_in(["STK000", "STK001", "STK002"]))

ma_gap = base.with_columns(
    **{
        f"gap_{n}d": col("close") / col("close").rolling_mean(n, order_by="ts", partition_by="symbol") - 1
        for n in WINDOWS
    }
)
print(ma_gap.sql())

SELECT *, "close" / avg("close") OVER (PARTITION BY "symbol" ORDER BY "ts" ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) - 1 AS "gap_5d", "close" / avg("close") OVER (PARTITION BY "symbol" ORDER BY "ts" ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) - 1 AS "gap_20d", "close" / avg("close") OVER (PARTITION BY "symbol" ORDER BY "ts" ROWS BETWEEN 59 PRECEDING AND CURRENT ROW) - 1 AS "gap_60d"
FROM "prices"
WHERE "symbol" IN ('STK000', 'STK001', 'STK002')


In [15]:
ma_gap.select("ts", "symbol", *[f"gap_{n}d" for n in WINDOWS]).sort(["ts", "symbol"]).to_pandas().tail(6)

,ts,symbol,gap_5d,gap_20d,gap_60d
1494,2024-11-28 20:00:00+00:00,STK000,0.000589,0.046048,0.064262
1495,2024-11-28 20:00:00+00:00,STK001,-0.002695,0.028779,0.034177
1496,2024-11-28 20:00:00+00:00,STK002,-0.019274,-0.018988,0.043117
1497,2024-11-29 20:00:00+00:00,STK000,0.014864,0.060963,0.083427
1498,2024-11-29 20:00:00+00:00,STK001,-0.017498,0.010656,0.018326
1499,2024-11-29 20:00:00+00:00,STK002,0.015216,0.017048,0.080730


クロスセクションの演算子は、*同じ瞬間の*仲間内で値を順位づけるので、比較する単位を引数に
取ります。z 化した signal をいくつか1つの合成値にまとめる、というのがファクター構築の形
そのものです。

In [16]:
combo = (
    db.table("prices")
    .with_columns(
        z_ret=(col("close") / col("open") - 1).cs_zscore(partition_by="ts"),
        z_vol=col("volume").cast("DOUBLE").cs_zscore(partition_by="ts"),
    )
    .with_columns(score=(col("z_ret") - col("z_vol")) / 2)
    .select("ts", "symbol", "z_ret", "z_vol", "score")
    .sort(["ts", "score"], descending=[False, True])
)
combo.to_pandas().head(5)

,ts,symbol,z_ret,z_vol,score
0,2023-01-02 20:00:00+00:00,STK034,1.935241,-0.691824,1.313533
1,2023-01-02 20:00:00+00:00,STK040,0.791453,-1.559526,1.175490
2,2023-01-02 20:00:00+00:00,STK021,0.728941,-1.457394,1.093167
3,2023-01-02 20:00:00+00:00,STK001,1.070606,-1.111451,1.091028
4,2023-01-02 20:00:00+00:00,STK005,0.683265,-1.265949,0.974607


## 5. バージョンのピン留めとジョイン

読み取り点はそのまま `db.table()` に渡って `h5i()` に落ちるので、ピン留めしたビルダの
クエリは、手書きの SQL と同じくソースの時点で束縛されます。

`.join()` は両側をサブクエリとして描き、`l` と `r` の別名を付けます。この別名が、特定の
側に手を伸ばすための約束事です。この2つが揃うと、「同じクエリを N 個のバージョンに対して」
という比較が関数呼び出しになります。

In [17]:
db.append("trades", cu.make_trades(symbols=["AAPL", "MSFT", "NVDA"], days=1, start="2026-06-04", seed=8))


def per_symbol(version=None):
    return db.table("trades", version=version).group_by("symbol").agg(
        count_star().alias("n"), col("ts").max().alias("last_ts")
    )


drift = per_symbol(1).join(per_symbol(), on="symbol").select(
    symbol=col("symbol", relation="l"),
    trades_added=col("n", relation="r") - col("n", relation="l"),
)
print(drift.sql())

SELECT "l"."symbol" AS "symbol", "r"."n" - "l"."n" AS "trades_added"
FROM (
  SELECT "symbol", count(*) AS "n", max("ts") AS "last_ts"
  FROM h5i('trades', 1)
  GROUP BY "symbol"
) AS "l"
INNER JOIN (
  SELECT "symbol", count(*) AS "n", max("ts") AS "last_ts"
  FROM "trades"
  GROUP BY "symbol"
) AS "r"
  ON "l"."symbol" = "r"."symbol"


In [18]:
drift.sort("symbol").to_pandas()

,symbol,trades_added
0,AAPL,24438
1,MSFT,14065
2,NVDA,22443


`.join_asof()` は `asof_join` テーブル関数に落ちます。この関数は*テーブル名*を取って両方を
最新で読むので、すでに動詞が適用された側やピン留めされた側を、ビルダは黙って無視せず
はっきり拒否します。絞り込みはジョインの後でやってください。

In [19]:
tape, quotes = cu.make_trades_and_quotes(days=2)  # shared base prices
for name, data in (("tape", tape), ("quotes", quotes)):
    db.create_table(name, data.schema, time_column="ts", sort_key=["ts", "symbol"])
    db.append(name, data)

try:
    db.table("tape").filter(col("symbol") == "AAPL").join_asof(db.table("quotes"), on="ts", by="symbol")
except h5i_db.InvalidInputError as e:
    print(f"{type(e).__name__}: {e}\nhint: {e.hint}")

InvalidInputError: [invalid_input] join_asof() needs a plain table on the left side, but operations have already been applied
hint: join first and filter afterwards, or materialise the side with .collect() and write it back


In [20]:
tq = (
    db.table("tape")
    .join_asof(db.table("quotes"), on="ts", by="symbol", tolerance=5_000_000)
    .filter(col("symbol") == "AAPL")
    .select("ts", "symbol", "price", "bid", "ask", mid=(col("bid") + col("ask")) / 2)
)
print(tq.sql())

SELECT "ts", "symbol", "price", "bid", "ask", ("bid" + "ask") / 2 AS "mid"
FROM asof_join('tape', 'quotes', 'ts', 'ts', 'symbol', 'backward', 5000000)
WHERE "symbol" = 'AAPL'


In [21]:
tq.to_pandas().head(4)

,ts,symbol,price,bid,ask,mid
0,2026-06-01 15:13:45.758336+00:00,AAPL,268.85,266.54,266.57,266.555
1,2026-06-01 15:13:46.306373+00:00,AAPL,268.82,266.54,266.58,266.560
2,2026-06-01 15:13:48.238865+00:00,AAPL,268.71,266.53,266.60,266.565
3,2026-06-01 15:13:50.079516+00:00,AAPL,268.78,266.48,266.55,266.515


## 6. 脱出口と、SQL へ戻る扉

動詞で SQL を全面的に覆うことは、意図的に目標にしていません。`sql_expr()` は、式が受け
付けられる場所ならどこにでも生の断片を落とします。テキストはそのまま入るので、クオートを
正しくするのが自分の仕事になる唯一の場所でもあります。

In [22]:
tails = (
    db.table("prices")
    .group_by("symbol")
    .agg(
        p01=sql_expr("approx_percentile_cont(close, 0.01)"),
        p99=sql_expr("approx_percentile_cont(close, 0.99)"),
    )
    .sort("symbol")
    .limit(4)
)
tails.to_pandas()

,symbol,p01,p99
0,STK000,25.7775,68.6225
1,STK001,172.5800,259.4775
2,STK002,209.8525,357.6050
3,STK003,110.8075,215.2300


いちばんよく手が伸びる脱出口は `lag` でしょう。`.lag()` メソッドはありませんが、`sql_expr`
の断片はウィンドウ化できるので、集約と同じように `.over()` を取ります。これで `lag`、
`lead`、`row_number` をはじめ、SQL のウィンドウ関数がひととおり使えます。日次リターンは
このクックブックでいちばん多く出てくる形です。

In [23]:
PREV_CLOSE = sql_expr("lag(close)").over(partition_by="symbol", order_by="ts")

rets = (
    db.table("prices")
    .with_columns(prev_close=PREV_CLOSE)
    .with_columns(ret=col("close") / col("prev_close") - 1)
    .filter(col("ret").is_not_null())
    .select("ts", "symbol", "ret")
)
print(rets.sql())

SELECT "ts", "symbol", "ret"
FROM (
  SELECT *, "close" / "prev_close" - 1 AS "ret"
  FROM (
    SELECT *, lag(close) OVER (PARTITION BY "symbol" ORDER BY "ts") AS "prev_close"
    FROM "prices"
  ) AS "_s1"
) AS "_s2"
WHERE "ret" IS NOT NULL


In [24]:
rets.sort(["ts", "symbol"]).to_pandas().head(4)

,ts,symbol,ret
0,2023-01-03 20:00:00+00:00,STK000,0.015468
1,2023-01-03 20:00:00+00:00,STK001,-0.001578
2,2023-01-03 20:00:00+00:00,STK002,0.025035
3,2023-01-03 20:00:00+00:00,STK003,0.015922


2段に分けた `with_columns` に注目してください。`ret` は1つ前の段が計算した `prev_close` を
読むので、ビルダは解決できない SQL を吐くかわりに階層を閉じます。断片を Python の名前に
一度だけ束ねて、ここでは `PREV_CLOSE` として、それを使い回す。ファクターライブラリを正直に
保つ習慣です。

パイプラインがビルダに収まらなくなったら、`.sql()` がクエリを渡してくれます。`db.sql()` に
貼って、そこから先を続けてください。2つの面は、真ん中に扉のある1つのシステムです。生成
される SQL は決定的なので、スナップショットテストや差分にもかけられます。

動詞がなく、文字列のほうが素直に読めるという理由で `db.sql()` に残るものもあります。
`UNION ALL`、深い多段の CTE、スカラサブクエリ、そして `gapfill`／`resample`／`tail` の
テーブル関数です。読み取り点を2つ積んでラベル付きの1つの結果にするのが、日常的な例です。

In [25]:
db.sql(
    """
    SELECT 'version 1' AS read_point, count(*) AS rows FROM h5i('trades', 1)
    UNION ALL
    SELECT 'latest',                  count(*)         FROM trades
    """
).to_pandas()

,read_point,rows
0,version 1,195277
1,latest,256223


## まとめ

- `db.table(...)` は遅延クエリです。動詞は新しいフレームを返し、`.collect()` や
  `.to_pandas()` まで何も走りません。`.sql()` がコンパイル結果の SQL を見せます。
- ビルダは `db.sql()` の上のコンパイラであって第2のエンジンではありません。セッションも
  テーブル関数も `h5i()` のバージョンピンも同じものです。
- 真偽の論理には `&`、`|`、`~` を使います。式が SQL の意味論を持つことも忘れずに。整数の
  `/` は切り捨てです。
- 手を伸ばすべきなのは、クエリを**生成する**とき、たとえばウィンドウや列をループで掃く
  ときです。f-string の SQL がクオートのバグを生む場面ですから。1度しか書かないクエリなら、
  素の SQL のほうが短いことも多いでしょう。
- `rolling_*` と `cs_*` のメソッドは本物の `PARTITION BY` を持ちます。全体の行ウィンドウを
  取るだけの `rolling_avg` 糖衣とは違います。
- `sql_expr()` が脱出口、`.sql()` が戻る扉です。どちらの面も二級市民ではありません。

In [26]:
db.close()